# Fact extraction and verification

This notebook reproduces the claim and verification outputs in §3.3 from the final raw model files.

In [1]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / "raw").is_dir():
    ROOT = ROOT.parent
RAW_AIOS = ROOT / "raw" / "aio_results"


def show_table(frame, formats=None):
    output = frame.copy()
    for column, template in (formats or {}).items():
        output[column] = output[column].map(template.format)
    display(HTML(output.to_html(index=False, border=0)))


LABELS = ["CLEAR", "VAGUE", "AMBIGUOUS", "INCORRECT", "OMITTED"]
claim_files = list(RAW_AIOS.rglob("claims_grok_4_1.json"))
verification_files = list(RAW_AIOS.rglob("verification_result_grok-4-1-fast-reasoning_v2.json"))

claims_per_aio = [
    len(json.loads(path.read_text(encoding="utf-8")).get("claims") or [])
    for path in claim_files
]

label_counts = Counter()
for path in verification_files:
    data = json.loads(path.read_text(encoding="utf-8"))
    for item in data.get("verifications") or []:
        label = str(item.get("label") or "").upper()
        if label in LABELS:
            label_counts[label] += 1

assert len(claim_files) == 7_583 and sum(claims_per_aio) == 98_098
assert len(verification_files) == 7_491 and sum(label_counts.values()) == 98_020
assert label_counts == Counter(CLEAR=82_933, VAGUE=4_271, AMBIGUOUS=1_367, INCORRECT=2_609, OMITTED=6_840)

## Table 2 — Final verification judgments

Consistent = Clear + Vague. Inconsistent = Ambiguous + Incorrect + Omitted.

In [2]:
table2 = pd.DataFrame({
    "Label": [label.title() for label in LABELS],
    "Claims": [label_counts[label] for label in LABELS],
})
table2["Share"] = 100 * table2["Claims"] / table2["Claims"].sum()
table2 = pd.concat([table2, pd.DataFrame([
    {"Label":"Consistent", "Claims":label_counts["CLEAR"]+label_counts["VAGUE"], "Share":100*(label_counts["CLEAR"]+label_counts["VAGUE"])/98_020},
    {"Label":"Inconsistent", "Claims":label_counts["AMBIGUOUS"]+label_counts["INCORRECT"]+label_counts["OMITTED"], "Share":100*(label_counts["AMBIGUOUS"]+label_counts["INCORRECT"]+label_counts["OMITTED"])/98_020},
])], ignore_index=True)
show_table(table2, {"Claims":"{:,}", "Share":"{:.1f}%"})

Label,Claims,Share
Clear,"82,933",84.6%
Vague,"4,271",4.4%
Ambiguous,"1,367",1.4%
Incorrect,"2,609",2.7%
Omitted,"6,840",7.0%
Consistent,"87,204",89.0%
Inconsistent,"10,816",11.0%


Human-validation statistics are not recalculated here because the annotation sheets are outside the current release scope.